# Experiment 013: CSQE â€” Corpus-Steered Query Expansion
## Model: Aya Expanse 8B | BM25 first-pass k=10 | 2 corpus + 2 blind expansions

**Algorithm:** CSQE (Corpus-Steered Query Expansion)  
**Reference:** arxiv 2402.18031  
**Key difference from Query2Doc:** LLM sees actual retrieved corpus documents before generating expansions.  
**Target:** Beat hybrid baseline (nDCG@10 = 0.6267)  
**Output:** `results/enhanced_queries/exp_013_csqe_aya_8b.pkl`


In [ ]:
# Clone repository
!git clone https://github.com/Osmanoor/graduation.git
%cd graduation/arabic-rag-query-enhancement

# Install Java 21 (required for Pyserini)
!apt-get install -qq openjdk-21-jdk-headless

# Install Python dependencies
!pip install -q pyserini faiss-cpu pytrec-eval transformers torch
!pip install -q datasets accelerate bitsandbytes huggingface_hub
!pip install -q bm25s PyStemmer nltk
!pip install -q --upgrade pillow

print('=' * 60)
print('Installation complete')
print('=' * 60)
print('IMPORTANT: Restart runtime now!')
print('  1. Runtime -> Restart runtime')
print('  2. Then run cells starting from Step 2 below')
print('=' * 60)


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to project
%cd /content/graduation/arabic-rag-query-enhancement

import os, sys
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# CRITICAL: Prevent TensorFlow from silently pre-allocating all GPU VRAM.
# Pyserini/BM25S imports can trigger TF in the background. Without this,
# TF grabs ~30GB on A100, leaving no room for the Aya model.
# This was NOT needed in the original Aya notebook because BM25 was loaded
# in a separate evaluation notebook -- but CSQE needs BM25 during generation.
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

# Setup symlinks for BM25S index (adjust drive_base if your path differs)
drive_base = '/content/drive/MyDrive/graduation project/colab_data'
!mkdir -p data/miracl_ar
!ln -sf "{drive_base}/bm25s_index" data/miracl_ar/bm25s_index
!ln -sf "{drive_base}/corpus_ids.pkl" data/miracl_ar/corpus_ids.pkl

print('Environment configured')

In [ ]:
from huggingface_hub import login

# Login to HuggingFace (prompted for token)
login()

print('Logged in to HuggingFace')
print('Make sure you have accepted the Aya Expanse license at:')
print('https://huggingface.co/CohereForAI/aya-expanse-8b')


In [ ]:
import os, sys, json, pickle, time, re
import numpy as np
import torch
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from src.utils.data_loader import MIRACLDataLoader

# NOTE: BM25SRetriever and evaluation are in a SEPARATE notebook
# (evaluate_enhanced_queries.ipynb), not here.

print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')


In [ ]:
CONFIG = {
    # BM25 first-pass
    'bm25_index_path': 'data/miracl_ar/bm25s_index',
    'bm25_corpus_ids_path': 'data/miracl_ar/corpus_ids.pkl',
    'top_k_docs': 5,               # Reduced from 10 -> 5 (top-5 carry most signal,
                                    # halves corpus prompt length -> faster prefill)
    'doc_truncation_tokens': 128,   # Truncate each retrieved doc to 128 tokens

    # LLM generation
    'model_name': 'CohereForAI/aya-expanse-8b',
    'temperature': 1.0,             # Diversity for multi-sample generation
    'max_new_tokens': 128,          # Reduced from 256 -> 128 (extractions are
                                    # ~200 tokens, blind passages ~150 tokens)
    'top_p': 0.9,
    'num_corpus_samples': 2,        # N1: corpus-originated expansions
    'num_blind_samples': 2,         # N2: blind (HyDE-style) expansions

    # Query construction
    # alpha = N_total = num_corpus + num_blind = 4
    'query_repetition': 4,

    # Batching (cross-query batching for speed on A100 80GB)
    'corpus_batch_size': 8,         # Batch size for corpus (long ~800-1000 tok) prompts
    'blind_batch_size': 32,         # Batch size for blind (short ~60-80 tok) prompts
    'max_corpus_prompt_length': 2048,  # Corpus prompts are long — do NOT truncate at 512
    'max_blind_prompt_length': 512,    # Blind prompts are short
    'macro_batch_size': 200,        # Queries per checkpoint cycle

    # Output
    'exp_id': 'exp_013',
    'output_pkl': 'results/enhanced_queries/exp_013_csqe_aya_8b.pkl',
    'output_trec_bm25': 'results/exp_013_csqe_bm25.txt',
    'run_name': 'exp_013_csqe_aya',
    'checkpoint_path': '/content/drive/MyDrive/exp_013_checkpoint.pkl',

    # Corpus dict
    'corpus_dict_drive_path': '/content/drive/MyDrive/graduation project/colab_data/corpus_dict.pkl',
}

print('CONFIG loaded:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
data_loader = MIRACLDataLoader(language='ar', split='dev')
topics, qrels = data_loader.load_all()

query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

print(f'Queries: {len(query_ids)}')
print(f'Qrels: {len(qrels)}')
print(f'Sample query: {query_texts[0]}')


In [ ]:
# Load MIRACL corpus for document text lookup
# BM25S index does NOT store original text (only tokenized form)
# We need the original text to pass retrieved docs to the LLM
#
# Uses direct JSONL.gz download (same approach as bm25s_baseline.ipynb)
# This avoids the UnicodeDecodeError from HuggingFace datasets loader

import gzip, requests

corpus_dict_path = CONFIG['corpus_dict_drive_path']

if os.path.exists(corpus_dict_path):
    print(f'Loading corpus_dict from Drive cache: {corpus_dict_path}')
    with open(corpus_dict_path, 'rb') as f:
        corpus_dict = pickle.load(f)
    print(f'Loaded {len(corpus_dict):,} documents from cache')
else:
    print('corpus_dict not cached on Drive. Downloading MIRACL corpus chunks...')
    base_url = "https://huggingface.co/datasets/miracl/miracl-corpus/resolve/main/miracl-corpus-v1.0-ar/docs-{}.jsonl.gz"
    num_chunks = 5

    corpus_dict = {}
    for chunk_idx in range(num_chunks):
        file_url = base_url.format(chunk_idx)
        temp_file = f'/tmp/miracl_ar_docs_{chunk_idx}.jsonl.gz'

        print(f'  Chunk {chunk_idx+1}/{num_chunks}: downloading...')
        response = requests.get(file_url, stream=True)
        with open(temp_file, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)

        print(f'  Chunk {chunk_idx+1}/{num_chunks}: parsing...')
        with gzip.open(temp_file, 'rt', encoding='utf-8') as f:
            for line in f:
                doc = json.loads(line)
                title = doc.get('title', '').strip()
                text = doc.get('text', '').strip()
                corpus_dict[doc['docid']] = f'{title} {text}'.strip() if title else text

        os.remove(temp_file)
        print(f'  Chunk {chunk_idx+1}/{num_chunks}: done ({len(corpus_dict):,} docs so far)')

    print('
Saving corpus_dict to Drive for future reuse...')
    os.makedirs(os.path.dirname(corpus_dict_path), exist_ok=True)
    with open(corpus_dict_path, 'wb') as f:
        pickle.dump(corpus_dict, f)
    print(f'Saved to {corpus_dict_path}')

print(f'
corpus_dict ready: {len(corpus_dict):,} documents')
sample_docid = list(corpus_dict.keys())[0]
print(f'Sample [{sample_docid}]: {corpus_dict[sample_docid][:100]}')

In [ ]:
# Load pre-computed BM25 first-pass results from Drive
# Original was k=10. We now use top_k_docs=5 for faster prompts.
# Just slice the top-5 from the cached k=10 results (no BM25S needed).

firstpass_path = '/content/drive/MyDrive/exp_013_firstpass.pkl'

if os.path.exists(firstpass_path):
    print(f'Loading first-pass results from {firstpass_path}')
    with open(firstpass_path, 'rb') as f:
        firstpass_results_raw = pickle.load(f)
    # Trim to top_k_docs (cached may have more)
    k = CONFIG['top_k_docs']
    firstpass_results = {
        qid: docs[:k] for qid, docs in firstpass_results_raw.items()
    }
    del firstpass_results_raw
    print(f'Loaded {len(firstpass_results)} queries, trimmed to top-{k} docs each')
else:
    raise FileNotFoundError(
        f'First-pass results not found at {firstpass_path}. '
        'Run the BM25 first-pass cell first (see notebook history).'
    )

# Verify
sample_qid = query_ids[0]
sample_docs = firstpass_results[sample_qid]
print(f'\nSample: qid={sample_qid}, {len(sample_docs)} docs')
print(f'  Top doc: {sample_docs[0]["docid"]} (score={sample_docs[0]["score"]:.4f})')
print(f'  Text preview: {sample_docs[0]["text"][:80]}')


In [ ]:
def parse_docid(docid):
    """Parse X#Y format to article_id and passage_position."""
    parts = docid.split('#')
    return {
        'article_id': int(parts[0]),
        'passage_pos': int(parts[1]) if len(parts) > 1 else 0
    }


def truncate_to_tokens(text, max_tokens=128, tokenizer=None):
    """
    Truncate text to approximately max_tokens.
    Uses tokenizer if available, else char-based fallback.
    """
    if not text:
        return ''
    if tokenizer is not None:
        tokens = tokenizer.encode(text, add_special_tokens=False)
        if len(tokens) > max_tokens:
            tokens = tokens[:max_tokens]
            return tokenizer.decode(tokens, skip_special_tokens=True)
        return text
    else:
        # Rough fallback: 128 tokens â‰ˆ 512 chars for Arabic
        return text[:512]


print('Helper functions defined')


In [ ]:
# SOURCE: Paper arxiv 2402.18031 Tables 1 & 2, Section 3.1
# English instructions (same pattern as all our existing Aya notebooks).
# Arabic output enforced via 'Respond in Arabic only.'

# â”€â”€ System prompts â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

BLIND_SYSTEM = (
    "You are asked to write a passage that answers the given query. "
    "Do not ask the user for further clarification. "
    "Respond in Arabic only."
)

CSQE_SYSTEM = (
    "You are an information retrieval assistant. "
    "You will examine retrieved documents and extract key sentences relevant to the query. "
    "The documents are in Arabic. Respond in Arabic only."
)

# â”€â”€ One-shot example (paper Table 2, English query/docs) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Aya 8B is multilingual â€” the English example generalises to Arabic queries.
CSQE_ONE_SHOT = (
    'Query: "how are some sharks warm blooded"\n'
    'Retrieved documents:\n'
    '1. Most sharks are cold-blooded. Some, like the Mako and the Great white shark, '
    'are partially warm-blooded (they are endotherms)...\n'
    '2. Are sharks cold-blooded or warm-blooded? Sharks have a reputation as cold-blooded...\n'
    '3. Great white sharks are some of the only warm blooded sharks...\n'
    'You will begin by examining the initially retrieved documents and identifying the ones '
    'that are relevant, even partially, to the query. Once the relevant documents are '
    'identified, you will extract the key sentences from each document that contribute '
    'to their relevance.\n'
    'Based on the query "how are some sharks warm blooded", I have examined the initially '
    'retrieved documents. Here are the relevant documents and the key sentences extracted '
    'from each:\n'
    'Document 1:\n'
    '"Most sharks are cold-blooded. Some, like the Mako and the Great white shark, '
    'are partially warm-blooded (they are endotherms)."\n'
    'Document 3:\n'
    '"Great white sharks are some of the only warm-blooded sharks."\n'
)


def build_csqe_prompt(query, retrieved_docs_truncated):
    """
    CSQE corpus-grounded prompt (paper Table 2).
    LLM should EXTRACT sentences from retrieved docs, not freely generate.
    """
    docs_str = '\n'.join(
        f'{i+1}. {doc}' for i, doc in enumerate(retrieved_docs_truncated)
    )
    prompt = (
        f'{CSQE_ONE_SHOT}\n'
        f'Query: "{query}"\n'
        f'Retrieved documents:\n{docs_str}\n'
        f'You will begin by examining the initially retrieved documents and identifying '
        f'the ones that are relevant, even partially, to the query. Once the relevant '
        f'documents are identified, you will extract the key sentences from each document '
        f'that contribute to their relevance. Respond in Arabic only.'
    )
    return prompt


def build_blind_prompt(query):
    """
    Blind (KEQE/Query2Doc) prompt â€” paper Table 1.
    System prompt handles the task; user message is just the query.
    """
    return query


print('Prompt templates defined')
print(f'BLIND_SYSTEM: {BLIND_SYSTEM[:60]}...')
print(f'CSQE_SYSTEM:  {CSQE_SYSTEM[:60]}...')


In [ ]:
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ── Clean up any leftover VRAM ──
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('=== GPU before model load ===')
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {torch.cuda.get_device_name(0)} ({total:.0f}GB)')

print(f'\nLoading Aya Expanse 8B in BF16 (no quantization)...')
print(f'Model: {CONFIG["model_name"]}')

# BF16: ~16GB VRAM. On A100 80GB this leaves ~64GB for KV cache + batching.
# Faster than 4-bit NF4 because no dequantization overhead.
# A100 has native BF16 tensor cores.
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # Required for decoder-only batch generation

model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True
)
model.eval()

print(f'\nAya Expanse 8B loaded (BF16)')
print(f'Model device: {model.device}')
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {allocated:.1f}GB / {total:.1f}GB used')
    print(f'Free for batching: {total - allocated:.1f}GB')

# Reload first-pass if needed (e.g., after runtime restart)
if 'firstpass_results' not in dir():
    fp_path = '/content/drive/MyDrive/exp_013_firstpass.pkl'
    print(f'\nReloading first-pass results from {fp_path}')
    with open(fp_path, 'rb') as f:
        firstpass_results = pickle.load(f)
    print(f'Loaded {len(firstpass_results)} queries')

In [ ]:
class CSQEEnhancer:
    """
    Corpus-Steered Query Expansion (CSQE) enhancer.
    Supports both sequential (for sanity checks) and batched (for full runs) generation.
    """

    def __init__(self, model, tokenizer, firstpass_results, config):
        self.model = model
        self.tokenizer = tokenizer
        self.firstpass = firstpass_results
        self.config = config

    def get_retrieved_docs(self, qid):
        """Get pre-computed BM25 first-pass docs, truncated."""
        raw_docs = self.firstpass.get(qid, [])
        docs = []
        for d in raw_docs:
            truncated = truncate_to_tokens(
                d['text'],
                max_tokens=self.config['doc_truncation_tokens'],
                tokenizer=self.tokenizer
            )
            docs.append({
                'docid': d['docid'],
                'score': d['score'],
                'text': truncated,
                'meta': parse_docid(d['docid'])
            })
        return docs

    # ── Sequential generation (for sanity checks) ──

    def generate_samples(self, system_prompt, user_prompt, n_samples, temperature=None):
        """
        Generate n_samples in ONE forward pass via num_return_sequences.
        Used for sanity checks (single query at a time).
        """
        if temperature is None:
            temperature = self.config['temperature']

        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ]

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        inputs = self.tokenizer(text, return_tensors='pt').to(self.model.device)
        input_len = inputs.input_ids.shape[1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.config['max_new_tokens'],
                temperature=temperature,
                top_p=self.config['top_p'],
                do_sample=True,
                num_return_sequences=n_samples,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        samples = []
        for seq in outputs:
            generated = seq[input_len:]
            decoded = self.tokenizer.decode(generated, skip_special_tokens=True).strip()
            samples.append(decoded)
        return samples

    def enhance(self, qid, query):
        """Sequential CSQE for a single query (sanity checks)."""
        retrieved_docs = self.get_retrieved_docs(qid)
        doc_texts = [d['text'] for d in retrieved_docs if d['text']]

        corpus_user_prompt = build_csqe_prompt(query, doc_texts)
        corpus_expansions = self.generate_samples(
            CSQE_SYSTEM, corpus_user_prompt,
            n_samples=self.config['num_corpus_samples'], temperature=1.0
        )

        blind_user_prompt = build_blind_prompt(query)
        blind_expansions = self.generate_samples(
            BLIND_SYSTEM, blind_user_prompt,
            n_samples=self.config['num_blind_samples'], temperature=1.0
        )

        alpha = self.config['query_repetition']
        all_expansions = corpus_expansions + blind_expansions
        final_query = (query + ' ') * alpha + ' '.join(all_expansions)

        return {
            'original': query,
            'retrieved_docids': [d['docid'] for d in retrieved_docs],
            'retrieved_doc_texts': [d['text'] for d in retrieved_docs],
            'corpus_expansions': corpus_expansions,
            'blind_expansions': blind_expansions,
            'enhanced': final_query,
        }

    # ── Batched generation (for full runs) ──

    def batch_generate(self, system_prompt, user_prompts, batch_size,
                       temperature=None, max_length=2048):
        """
        Generate 1 sample per prompt for a list of prompts, in mini-batches.
        Same pattern as our proven Aya Query2Doc notebook (enhance_batch_parallel).
        Returns list of decoded strings, one per prompt.
        """
        if temperature is None:
            temperature = self.config['temperature']

        # Step 1: build chat-formatted texts (not batchable — loop)
        texts = []
        for user_prompt in user_prompts:
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_prompt},
            ]
            text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )
            texts.append(text)

        # Step 2: process in mini-batches
        all_outputs = []
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]

            inputs = self.tokenizer(
                batch_texts,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=max_length,
            ).to(self.model.device)

            input_length = inputs.input_ids.shape[1]

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=self.config['max_new_tokens'],
                    temperature=temperature,
                    top_p=self.config['top_p'],
                    do_sample=True,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )

            decoded = self.tokenizer.batch_decode(
                outputs[:, input_length:],
                skip_special_tokens=True
            )
            all_outputs.extend([d.strip() for d in decoded])

        return all_outputs

    def batch_enhance(self, qids, queries):
        """
        Full CSQE pipeline for a batch of queries.
        4 batched generate passes: corpus×2 + blind×2.
        Returns list of result dicts.
        """
        n = len(qids)
        corpus_bs = self.config['corpus_batch_size']
        blind_bs = self.config['blind_batch_size']
        max_corpus_len = self.config['max_corpus_prompt_length']
        max_blind_len = self.config['max_blind_prompt_length']

        # ── Step 1: Build all prompts ──
        corpus_prompts = []
        blind_prompts = []
        all_retrieved = []
        for qid, query in zip(qids, queries):
            docs = self.get_retrieved_docs(qid)
            all_retrieved.append(docs)
            doc_texts = [d['text'] for d in docs if d['text']]
            corpus_prompts.append(build_csqe_prompt(query, doc_texts))
            blind_prompts.append(build_blind_prompt(query))

        # ── Step 2: Corpus expansions (2 passes for 2 samples) ──
        corpus_samples_1 = self.batch_generate(
            CSQE_SYSTEM, corpus_prompts, corpus_bs,
            temperature=1.0, max_length=max_corpus_len
        )
        corpus_samples_2 = self.batch_generate(
            CSQE_SYSTEM, corpus_prompts, corpus_bs,
            temperature=1.0, max_length=max_corpus_len
        )

        # ── Step 3: Blind expansions (2 passes for 2 samples) ──
        blind_samples_1 = self.batch_generate(
            BLIND_SYSTEM, blind_prompts, blind_bs,
            temperature=1.0, max_length=max_blind_len
        )
        blind_samples_2 = self.batch_generate(
            BLIND_SYSTEM, blind_prompts, blind_bs,
            temperature=1.0, max_length=max_blind_len
        )

        # ── Step 4: Assemble results ──
        alpha = self.config['query_repetition']
        results = []
        for i in range(n):
            corpus_exps = [corpus_samples_1[i], corpus_samples_2[i]]
            blind_exps = [blind_samples_1[i], blind_samples_2[i]]
            all_exps = corpus_exps + blind_exps
            final = (queries[i] + ' ') * alpha + ' '.join(all_exps)

            results.append({
                'qid': qids[i],
                'original': queries[i],
                'retrieved_docids': [d['docid'] for d in all_retrieved[i]],
                'retrieved_doc_texts': [d['text'] for d in all_retrieved[i]],
                'corpus_expansions': corpus_exps,
                'blind_expansions': blind_exps,
                'enhanced': final,
            })

        return results


enhancer = CSQEEnhancer(
    model=model, tokenizer=tokenizer,
    firstpass_results=firstpass_results, config=CONFIG
)

print('CSQEEnhancer ready')
print(f'  Sequential: enhance(qid, query) — for sanity checks')
print(f'  Batched: batch_enhance(qids, queries) — for full run')
print(f'  Corpus batch size: {CONFIG["corpus_batch_size"]}')
print(f'  Blind batch size: {CONFIG["blind_batch_size"]}')

In [ ]:
# Sanity check: sequential (5 queries) then batched (same 5) — compare outputs

print('=== Sequential sanity check (5 queries) ===\n')
sanity_results_seq = []
sanity_start = time.time()

for qid in query_ids[:5]:
    q = topics[qid]['title']
    t0 = time.time()
    result = enhancer.enhance(qid, q)
    dt = time.time() - t0
    result['qid'] = qid
    sanity_results_seq.append(result)

    print(f'QID: {qid} ({dt:.1f}s)')
    print(f'  Query: {q}')
    print(f'  Corpus exp 1 ({len(result["corpus_expansions"][0])}ch): '
          f'{result["corpus_expansions"][0][:120]}')
    print(f'  Blind exp 1  ({len(result["blind_expansions"][0])}ch): '
          f'{result["blind_expansions"][0][:120]}')
    print()

seq_elapsed = time.time() - sanity_start
per_query_seq = seq_elapsed / 5
print(f'Sequential: {seq_elapsed:.0f}s total, {per_query_seq:.1f}s/query')
print(f'  (Full run would take: {per_query_seq * len(query_ids) / 3600:.1f}h)')

# Now test batched on the same 5 queries
print('\n=== Batched sanity check (same 5 queries) ===\n')
batch_start = time.time()
sanity_results_batch = enhancer.batch_enhance(
    query_ids[:5],
    [topics[qid]['title'] for qid in query_ids[:5]]
)
batch_elapsed = time.time() - batch_start
per_query_batch = batch_elapsed / 5

for r in sanity_results_batch:
    print(f'QID: {r["qid"]}')
    print(f'  Corpus exp 1 ({len(r["corpus_expansions"][0])}ch): '
          f'{r["corpus_expansions"][0][:120]}')
    print(f'  Blind exp 1  ({len(r["blind_expansions"][0])}ch): '
          f'{r["blind_expansions"][0][:120]}')
    print()

print(f'Batched: {batch_elapsed:.0f}s total, {per_query_batch:.1f}s/query')
print(f'Speedup: {per_query_seq / per_query_batch:.1f}x')
print(f'Estimated full run (batched): {per_query_batch * len(query_ids) / 3600:.1f}h')

if torch.cuda.is_available():
    print(f'\nVRAM: {torch.cuda.memory_allocated()/1024**3:.1f}GB / '
          f'{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB')

In [ ]:
# Full batched generation with checkpointing

start_idx = 0
results = []

# Resume from checkpoint if exists
if os.path.exists(CONFIG['checkpoint_path']):
    print(f'Checkpoint found: {CONFIG["checkpoint_path"]}')
    with open(CONFIG['checkpoint_path'], 'rb') as f:
        checkpoint = pickle.load(f)
    results = checkpoint['results']
    start_idx = len(results)
    print(f'Resuming from query {start_idx}/{len(query_ids)}')
else:
    print('No checkpoint found, starting from scratch')

remaining_qids = query_ids[start_idx:]
remaining_texts = query_texts[start_idx:]
macro_bs = CONFIG['macro_batch_size']

total_remaining = len(remaining_qids)
print(f'\nCSQE batched generation: {total_remaining} queries remaining')
print(f'  Macro-batch size: {macro_bs}')
print(f'  Corpus mini-batch: {CONFIG["corpus_batch_size"]}')
print(f'  Blind mini-batch: {CONFIG["blind_batch_size"]}')
print('=' * 60)

start_time = time.time()

for batch_start in range(0, total_remaining, macro_bs):
    batch_end = min(batch_start + macro_bs, total_remaining)
    batch_qids = remaining_qids[batch_start:batch_end]
    batch_queries = remaining_texts[batch_start:batch_end]
    batch_num = batch_start // macro_bs + 1
    total_batches = (total_remaining + macro_bs - 1) // macro_bs

    print(f'\nBatch {batch_num}/{total_batches} ({len(batch_qids)} queries)...')

    try:
        batch_results = enhancer.batch_enhance(batch_qids, batch_queries)
        results.extend(batch_results)
    except Exception as e:
        print(f'  Batch error: {e}')
        print(f'  Falling back to sequential for this batch...')
        for qid, query in zip(batch_qids, batch_queries):
            try:
                result = enhancer.enhance(qid, query)
                result['qid'] = qid
                results.append(result)
            except Exception as e2:
                print(f'    ERROR qid={qid}: {e2}')
                alpha = CONFIG['query_repetition']
                results.append({
                    'qid': qid,
                    'original': query,
                    'retrieved_docids': [],
                    'retrieved_doc_texts': [],
                    'corpus_expansions': [''] * CONFIG['num_corpus_samples'],
                    'blind_expansions': [''] * CONFIG['num_blind_samples'],
                    'enhanced': (query + ' ') * alpha,
                    'error': str(e2)
                })

    # Checkpoint after each macro-batch
    with open(CONFIG['checkpoint_path'], 'wb') as f:
        pickle.dump({'results': results, 'config': CONFIG}, f)

    elapsed = time.time() - start_time
    done = len(results) - start_idx
    rate = done / (elapsed / 60) if elapsed > 0 else 0
    remaining_q = len(query_ids) - len(results)
    eta_min = remaining_q / rate if rate > 0 else 0
    print(f'  Done: {len(results)}/{len(query_ids)} | '
          f'{elapsed/60:.1f}min elapsed | {rate:.0f} q/min | '
          f'~{eta_min:.0f}min remaining')

elapsed = time.time() - start_time
print(f'\n{"=" * 60}')
print(f'COMPLETE: {len(results)} queries in {elapsed/60:.1f} minutes')
print(f'Rate: {len(results) / (elapsed/60):.1f} queries/minute')

errors = [r for r in results if 'error' in r]
if errors:
    print(f'Errors: {len(errors)} queries fell back to no-expansion')

In [ ]:
# Save in SAME format as all other enhanced_queries pkl files
# evaluate_enhanced_queries.ipynb reads: 'query_ids', 'original', 'enhanced'

enhanced_queries_flat = [r['enhanced'] for r in results]

output = {
    'query_ids': [r['qid'] for r in results],
    'original': [topics[r['qid']]['title'] for r in results],
    'enhanced': enhanced_queries_flat,
    'model': CONFIG['model_name'],
    'config': CONFIG,
    'stats': {
        'total_queries': len(results),
        'avg_original_len': float(np.mean([len(r['original']) for r in results])),
        'avg_enhanced_len': float(np.mean([len(r['enhanced']) for r in results])),
        'avg_expansion_ratio': float(np.mean([
            len(r['enhanced']) / len(r['original'])
            for r in results if len(r['original']) > 0
        ])),
        'avg_corpus_exp_len': float(np.mean([
            np.mean([len(e) for e in r['corpus_expansions']])
            for r in results if r['corpus_expansions']
        ])),
        'avg_blind_exp_len': float(np.mean([
            np.mean([len(e) for e in r['blind_expansions']])
            for r in results if r['blind_expansions']
        ])),
        'error_count': sum(1 for r in results if 'error' in r),
    },
    'full_results': results,
}

print('Stats:')
for k, v in output['stats'].items():
    print(f'  {k}: {v}')

# Save to Google Drive
drive_pkl = '/content/drive/MyDrive/exp_013_csqe_aya_8b.pkl'
with open(drive_pkl, 'wb') as f:
    pickle.dump(output, f)
print(f'\nSaved to: {drive_pkl}')
print(f'File size: {os.path.getsize(drive_pkl) / 1024**2:.1f} MB')

# Also save locally
import shutil
os.makedirs('results/enhanced_queries', exist_ok=True)
shutil.copy(drive_pkl, CONFIG['output_pkl'])
print(f'Copied to: {CONFIG["output_pkl"]}')

print('\n=== DONE ===')
print(f'Evaluate this pkl in evaluate_enhanced_queries.ipynb')
print(f'Upload: {drive_pkl}')


In [ ]:
# === EVALUATION ===
# Run evaluation in the SEPARATE evaluation notebook:
#   experiments/evaluate_enhanced_queries.ipynb
#
# 1. Upload the pkl: /content/drive/MyDrive/exp_013_csqe_aya_8b.pkl
# 2. The evaluation notebook computes: nDCG@10, Recall@10, Recall@100, MRR
#
# Target: nDCG@10 > 0.6267 (hybrid baseline)
#
# After evaluation, update:
#   - CLAUDE.md reference baselines table
#   - TASKS.md (mark 6.3b-implement done)
#   - Thesis Chapter 4

print('Notebook complete. Run evaluation separately.')
print(f'PKL saved to: /content/drive/MyDrive/exp_013_csqe_aya_8b.pkl')


In [ ]:
# Baselines for reference (fill in after evaluation):
#
# BM25 baseline (no QE):           0.4621 nDCG@10
# mDPR baseline (no QE):           0.4993
# Aya blind Query2Doc (BM25):      0.5046
# Aya blind Q2D beta=2 (BM25):     0.5855
# Hybrid RRF baseline (no QE):     0.6267  <-- target to beat
# CSQE Aya 8B (this exp):          ???     <-- run evaluation notebook

print('Fill in after running evaluation notebook')


In [ ]:
# Qualitative analysis: for 20 random queries, inspect
#   - Retrieved docs (are they relevant?)
#   - Corpus expansion (is it grounded in retrieved docs?)
#   - Blind expansion (is it hallucinating?)
# This feeds the thesis qualitative analysis section.

import random
random.seed(42)

sample_indices = random.sample(range(len(results)), min(20, len(results)))

qualitative_rows = []

for idx in sample_indices:
    r = results[idx]
    qid = r['qid']
    row = {
        'qid': qid,
        'query': r['original'],
        'top_docid': r['retrieved_docids'][0] if r['retrieved_docids'] else '',
        'top_doc_text': r['retrieved_doc_texts'][0][:200] if r['retrieved_doc_texts'] else '',
        'corpus_exp_1': r['corpus_expansions'][0][:200] if r['corpus_expansions'] else '',
        'blind_exp_1': r['blind_expansions'][0][:200] if r['blind_expansions'] else '',
    }
    qualitative_rows.append(row)

# Print first 5 in notebook
for row in qualitative_rows[:5]:
    print(f"{'=' * 60}")
    print(f'QID: {row["qid"]}')
    print(f'Query: {row["query"]}')
    print(f'Top retrieved doc [{row["top_docid"]}]:')
    print(f'  {row["top_doc_text"]}')
    print(f'Corpus exp 1 (should quote from doc above):')
    print(f'  {row["corpus_exp_1"]}')
    print(f'Blind exp 1 (should be free-form passage):')
    print(f'  {row["blind_exp_1"]}')
    print()

# Save all 20 to Drive for thesis analysis
qualitative_path = '/content/drive/MyDrive/exp_013_qualitative_analysis.json'
with open(qualitative_path, 'w', encoding='utf-8') as f:
    json.dump(qualitative_rows, f, ensure_ascii=False, indent=2)
print(f'Qualitative analysis saved to: {qualitative_path}')

# Summary: % of corpus expansions that are non-empty
non_empty_corpus = sum(
    1 for r in results
    if r['corpus_expansions'] and any(len(e) > 20 for e in r['corpus_expansions'])
)
print(f'\nCorpus expansions non-empty: {non_empty_corpus}/{len(results)} '
      f'({100*non_empty_corpus/len(results):.1f}%)')

# Next steps based on result:
# If nDCG@10 > 0.6267: Run ablations exp_013c (corpus-only) and exp_013d (blind-only)
#   to prove corpus grounding adds value for the thesis.
# If nDCG@10 < 0.6267: Try exp_013b (k=15) or exp_013e (alpha=2) first.
# Run exp_013f (Jais-2-8B) regardless â€” Jais-2 was best in Phase 2.
